In [1]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset, DatasetInfo
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Microbiology events for all cohort patients across their full relevant stay window:
#   - ED-only patients (no hadm_id): ED visit window only (intime → outtime)
#   - Admitted, no ICU: full hospital stay (ed_intime → dischtime)
#   - Admitted, went to ICU: ED intime up to ICU transfer (ed_intime → first_icu_intime)
# result_available_in_er flags the ~1.2% of cultures whose results came back before ED discharge

query_micro = """
SELECT
  c.subject_id,
  c.hadm_id,
  c.ed_stay_id,
  m.microevent_id,
  m.micro_specimen_id,
  m.charttime,
  m.spec_itemid,
  m.spec_type_desc,
  m.test_itemid,
  m.test_name,
  m.org_name,
  m.comments,
  m.storetime,
  IF(m.storetime IS NOT NULL
     AND CAST(m.storetime AS DATETIME) < c.ed_outtime, 1, 0) AS result_available_in_er,
  c.disposition,
  c.cohort_label
FROM `ads-599-final.rl_project.cohort_base` c
INNER JOIN `physionet-data.mimiciv_3_1_hosp.microbiologyevents` m
  ON c.subject_id = m.subject_id
WHERE m.charttime IS NOT NULL
  AND m.charttime >= c.ed_intime
  AND m.charttime < CASE
    -- Admitted patients who went to ICU: cut off at ICU transfer
    WHEN c.hadm_id IS NOT NULL AND c.first_icu_intime IS NOT NULL
      THEN c.first_icu_intime
    -- Admitted patients who never went to ICU: full hospital stay
    WHEN c.hadm_id IS NOT NULL AND c.first_icu_intime IS NULL
      THEN c.dischtime
    -- ED-only patients: ED window only
    ELSE c.ed_outtime
  END
ORDER BY c.subject_id, c.ed_stay_id, m.charttime
"""

df_micro = client.query(query_micro).to_dataframe()

admitted = df_micro[df_micro['hadm_id'].notna()]
home     = df_micro[df_micro['hadm_id'].isna()]

print(f"Total rows          : {len(df_micro):,}")
print(f"Unique ED stays     : {df_micro['ed_stay_id'].nunique():,}")
print(f"Unique subjects     : {df_micro['subject_id'].nunique():,}")
print()
print(f"Admitted rows       : {len(admitted):,}  |  unique stays: {admitted['ed_stay_id'].nunique():,}")
print(f"Discharged home rows: {len(home):,}  |  unique stays: {home['ed_stay_id'].nunique():,}")
print()
rows_per_stay = df_micro.groupby('ed_stay_id').size()
print(f"Stays with 1 culture row  : {(rows_per_stay == 1).sum():,}")
print(f"Stays with 2+ culture rows: {(rows_per_stay > 1).sum():,}")
print(f"Max culture rows per stay : {rows_per_stay.max():,}")
print()
print(f"Result available in ER: {df_micro['result_available_in_er'].sum():,} ({df_micro['result_available_in_er'].mean()*100:.1f}%)")
df_micro.head(20)

Total rows          : 842,611
Unique ED stays     : 174,901
Unique subjects     : 96,961

Admitted rows       : 726,335  |  unique stays: 122,813
Discharged home rows: 116,276  |  unique stays: 52,088

Stays with 1 culture row  : 70,039
Stays with 2+ culture rows: 104,862
Max culture rows per stay : 248

Result available in ER: 9,879 (1.2%)


,subject_id,hadm_id,ed_stay_id,microevent_id,micro_specimen_id,charttime,spec_itemid,spec_type_desc,test_itemid,test_name,org_name,comments,storetime,result_available_in_er,disposition,cohort_label
0,10000032,29079034,32952584,29,9682545,2180-07-23 06:39:00,70012,BLOOD CULTURE,90201,"Blood Culture, Routine",NaN,NO GROWTH.,2180-07-29 07:37:00,0,HOME,ED_DISCHARGE_RETURN_ICU
1,10000032,29079034,32952584,30,9150362,2180-07-23 06:50:00,70079,URINE,90039,URINE CULTURE,NaN,"<10,000 organisms/ml.",2180-07-24 11:53:00,0,HOME,ED_DISCHARGE_RETURN_ICU
2,10000032,22595853,33258284,13,5674338,2180-05-06 22:25:00,70012,BLOOD CULTURE,90201,"Blood Culture, Routine",NaN,NO GROWTH.,2180-05-12 08:07:00,0,ADMITTED,ED_WARD_DISCHARGE
3,10000032,22595853,33258284,14,1758637,2180-05-07 00:10:00,70079,URINE,90039,URINE CULTURE,NaN,"MIXED BACTERIAL FLORA ( >= 3 COLONY TYPES), CO...",2180-05-08 11:12:00,0,ADMITTED,ED_WARD_DISCHARGE
4,10000032,22595853,33258284,15,8440864,2180-05-07 00:19:00,70070,SWAB,90115,R/O VANCOMYCIN RESISTANT ENTEROCOCCUS,NaN,No VRE isolated.,2180-05-09 07:57:00,0,ADMITTED,ED_WARD_DISCHARGE
5,10000032,22595853,33258284,18,780339,2180-05-07 10:11:00,70053,PERITONEAL FLUID,90245,ANAEROBIC CULTURE,NaN,NO GROWTH.,2180-05-13 13:34:00,0,ADMITTED,ED_WARD_DISCHARGE
6,10000032,22595853,33258284,16,780339,2180-05-07 10:11:00,70053,PERITONEAL FLUID,90270,GRAM STAIN,NaN,2+ (1-5 per 1000X FIELD): POLYMORPHONUCLEA...,2180-05-07 15:13:00,0,ADMITTED,ED_WARD_DISCHARGE
7,10000032,22595853,33258284,17,780339,2180-05-07 10:11:00,70053,PERITONEAL FLUID,90268,FLUID CULTURE,NaN,NO GROWTH.,2180-05-10 10:39:00,0,ADMITTED,ED_WARD_DISCHARGE
8,10000032,25742920,35968195,34,118736,2180-08-06 00:30:00,70053,PERITONEAL FLUID,90268,FLUID CULTURE,NaN,NO GROWTH.,2180-08-09 10:18:00,0,ADMITTED,ED_WARD_DISCHARGE
9,10000032,25742920,35968195,35,118736,2180-08-06 00:30:00,70053,PERITONEAL FLUID,90245,ANAEROBIC CULTURE,NaN,NO GROWTH.,2180-08-12 14:53:00,0,ADMITTED,ED_WARD_DISCHARGE


In [4]:
PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

info = DatasetInfo(
    description=(
        "Microbiology culture events from hosp.microbiologyevents for cohort patients during their stay window. "
        "Includes culture order time, specimen type, organism name, and antibiotic sensitivity results. "
        "culture_ordered is the real-time state signal; culture_positive is a retrospective label only "
        "(~2% of results available before ED discharge)."
    ),
    license=PHYSIONET_LICENSE,
)

ds = Dataset.from_pandas(df_micro, split='microbiology_events', info=info)
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="microbiology_events", data_dir="microbiology")
print("Pushed microbiology_events to HuggingFace Hub.")

Setting num_proc from 1 back to 1 for the microbiology_events split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed microbiology_events to HuggingFace Hub.
